#**Code Refinement**

In [ ]:
# Cloning the repository in the notebook
import os

REPO_OWNER = "Epoch-2026-2027"
REPO_NAME = "Learning_Phase"
BRANCH_NAME = "Utkarsh"

REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"Directory '{REPO_NAME}' already exists.")

os.chdir(REPO_NAME)
print(f"Current working directory changed to: {os.getcwd()}")

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Importing the necessary libraries

import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from tokenizers import Tokenizer
from tokenizers. models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

##**Data Pre-processing**
The pre-processing code has been commented out since it only had to be run once. The encoded text sets were saved and are then loaded into the notebook.

In [ ]:
"""
# Loading the training, validation and test datasets

train_data = pd.read_parquet('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/train-00000-of-00001.parquet')
val_data = pd.read_parquet('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/validation-00000-of-00001.parquet')
test_data = pd.read_parquet('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/test-00000-of-00001.parquet')
"""

In [ ]:
"""
# Splitting each of them into the X (buggy) and y (fixed) code

X_train = train_data['buggy'].copy()
y_train = train_data['fixed'].copy()

X_val = val_data['buggy'].copy()
y_val = val_data['fixed'].copy()

X_test = test_data['buggy'].copy()
y_test = test_data['fixed'].copy()
"""

In [ ]:
"""
# Tokenizing, Encoding and Padding the Text

# LLM used here for help with the syntax to set up
# the tokenizer and train it
# Initializing a word-level (whitespace) tokenizer
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = WordLevelTrainer(
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"],
    vocab_size=10000
)

# Training the tokenizer on the training set
training_text_list = train_data[['buggy', 'fixed']].melt()['value'].astype(str).tolist()
tokenizer.train_from_iterator(training_text_list, trainer=trainer)

# Using [CLS] and [SEP] as the BOS and EOS tokens, respectively
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[CLS]",
    eos_token="[SEP]"
)
"""

In [ ]:
"""
# Saving the tokenizer
tokenizer.save_pretrained("./code_refinement_tokenizer")
"""

In [ ]:
"""
# Tokenizing for the Recurrent Architectures

# Tokenizing all the text sets. Here, the max_length is chosen after experimentation,
# such that the number of unnecessary padding tokens can be minimized

X_train = np.array(X_train.tolist())
y_train = np.array(y_train.tolist())
X_val = np.array(X_val.tolist())
y_val = np.array(y_val.tolist())
X_test = np.array(X_test.tolist())
y_test = np.array(y_test.tolist())

# Setting the padding side to left for the inputs X
# Not adding the BOS and EOS tokens
tokenizer.padding_side = 'left'

X_train = tokenizer(
    X_train.tolist(),
    padding='max_length',
    truncation=True,
    max_length=220,
    add_special_tokens=False,
    return_tensors='np'
)

X_val = tokenizer(
    X_val.tolist(),
    padding='max_length',
    truncation=True,
    max_length=220,
    add_special_tokens=False,
    return_tensors='np'
)

X_test = tokenizer(
    X_test.tolist(),
    padding='max_length',
    truncation=True,
    max_length=220,
    add_special_tokens=False,
    return_tensors='np'
)

# Setting the padding side to right for the outputs y
# Adding the BOS and EOS tokens
tokenizer.padding_side = 'right'

y_train = [f"{tokenizer.bos_token} {text} {tokenizer.eos_token}" for text in y_train]
y_train = tokenizer(
    y_train,
    padding='max_length',
    truncation=True,
    max_length=220,
    return_tensors='np'
)

y_val = [f"{tokenizer.bos_token} {text} {tokenizer.eos_token}" for text in y_val]
y_val = tokenizer(
    y_val,
    padding='max_length',
    truncation=True,
    max_length=220,
    return_tensors='np'
)

y_test = [f"{tokenizer.bos_token} {text} {tokenizer.eos_token}" for text in y_test]
y_test = tokenizer(
    y_test,
    padding='max_length',
    truncation=True,
    max_length=220,
    return_tensors='np'
)
"""

In [ ]:
"""
# Tokenizing for the Transformer Architecture

# Tokenizing all the text sets. Here, the max_length is chosen after experimentation,
# such that the number of unnecessary padding tokens can be minimized

X_train = np.array(X_train.tolist())
y_train = np.array(y_train.tolist())
X_val = np.array(X_val.tolist())
y_val = np.array(y_val.tolist())
X_test = np.array(X_test.tolist())
y_test = np.array(y_test.tolist())

# Setting the padding side to right for the inputs X and outputs y
# Not adding the BOS and EOS tokens
tokenizer.padding_side = 'right'

X_train_transformer = tokenizer(
    X_train.tolist(),
    padding='max_length',
    truncation=True,
    max_length=220,
    add_special_tokens=False,
    return_tensors='np'
)

X_val_transformer = tokenizer(
    X_val.tolist(),
    padding='max_length',
    truncation=True,
    max_length=220,
    add_special_tokens=False,
    return_tensors='np'
)

X_test_transformer = tokenizer(
    X_test.tolist(),
    padding='max_length',
    truncation=True,
    max_length=220,
    add_special_tokens=False,
    return_tensors='np'
)

# Adding the BOS and EOS tokens
y_train = [f"{tokenizer.bos_token} {text} {tokenizer.eos_token}" for text in y_train]
y_train_transformer = tokenizer(
    y_train,
    padding='max_length',
    truncation=True,
    max_length=220,
    return_tensors='np'
)

y_val = [f"{tokenizer.bos_token} {text} {tokenizer.eos_token}" for text in y_val]
y_val_transformer = tokenizer(
    y_val,
    padding='max_length',
    truncation=True,
    max_length=220,
    return_tensors='np'
)

y_test = [f"{tokenizer.bos_token} {text} {tokenizer.eos_token}" for text in y_test]
y_test_transformer = tokenizer(
    y_test,
    padding='max_length',
    truncation=True,
    max_length=220,
    return_tensors='np'
)
"""

In [ ]:
"""
# Saving all the encoded text sets to the drive for later use

# For the Recurrent Architectures
X_train = X_train['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/train_buggy_encodings.npz', train_buggy_encodings=X_train)

y_train = y_train['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/train_fixed_encodings.npz', train_fixed_encodings=y_train)

X_val = X_val['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/val_buggy_encodings.npz', val_buggy_encodings=X_val)

y_val = y_val['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/val_fixed_encodings.npz', val_fixed_encodings=y_val)

X_test = X_test['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/test_buggy_encodings.npz', test_buggy_encodings=X_test)

y_test = y_test['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/test_fixed_encodings.npz', test_fixed_encodings=y_test)
"""

In [ ]:
"""
# For the Transformer Architecture
X_train_transformer = X_train_transformer['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/train_transformer_buggy_encodings.npz', train_transformer_buggy_encodings=X_train_transformer)

y_train_transformer = y_train_transformer['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/train_transformer_fixed_encodings.npz', train_transformer_fixed_encodings=y_train_transformer)

X_val_transformer = X_val_transformer['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/val_transformer_buggy_encodings.npz', val_transformer_buggy_encodings=X_val_transformer)

y_val_transformer = y_val_transformer['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/val_transformer_fixed_encodings.npz', val_transformer_fixed_encodings=y_val_transformer)

X_test_transformer = X_test_transformer['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/test_transformer_buggy_encodings.npz', test_transformer_buggy_encodings=X_test_transformer)

y_test_transformer = y_test_transformer['input_ids']
np.savez_compressed('/content/drive/MyDrive/Datasets/Epoch/Code_Refinement_Dataset/test_transformer_fixed_encodings.npz', test_transformer_fixed_encodings=y_test_transformer)
"""

##**Baseline Model (LSTM):**

In [ ]:
# Loading all the encoded text sets

X_train = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/train_buggy_encodings.npz')
X_train = X_train['train_buggy_encodings']

y_train = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/train_fixed_encodings.npz')
y_train = y_train['train_fixed_encodings']

X_val = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/val_buggy_encodings.npz')
X_val = X_val['val_buggy_encodings']

y_val = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/val_fixed_encodings.npz')
y_val = y_val['val_fixed_encodings']

X_test = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/test_buggy_encodings.npz')
X_test = X_test['test_buggy_encodings']

y_test = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/test_fixed_encodings.npz')
y_test = y_test['test_fixed_encodings']

In [ ]:
# Finding the vocab size for the architecture

vocab_size = int(max(X_train.max(), y_train.max())) + 1

In [ ]:
# Wrapping the text data into a PyTorch Dataset

class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).long()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

# Creating the DataLoader instances for the training, validation and
# testing sets
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader = DataLoader(TextDataset(X_val, y_val), batch_size=128, shuffle=False)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=128, shuffle=False)

In [ ]:
# Baseline LSTM Architecture

class LSTM(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
    super().__init__()

    # Initializing the LSTM parameters
    self.vocab_size = vocab_size
    self.embedding_dim = embedding_dim
    self.hidden_dim = hidden_dim
    self.output_dim = output_dim

    # Embedding layer for the encoder to project the input to the embedding space
    self.encoder_embedding = nn.Embedding(self.vocab_size, self.embedding_dim)

    # Using a unidirectional encoder LSTM with 1 layer
    self.encoder = nn.LSTM(
        input_size=self.embedding_dim,
        hidden_size=self.hidden_dim,
        num_layers=1,
        batch_first=True,
        bidirectional=False
    )

    # Embedding layer for the decoder to project the expected output to the embedding space
    self.decoder_embedding = nn.Embedding(self.vocab_size, self.embedding_dim)

    # Creating the decoder LSTM with 1 layer
    self.decoder = nn.LSTM(
        input_size=self.embedding_dim,
        hidden_size=self.hidden_dim,
        num_layers=1,
        batch_first=True,
        bidirectional=False
    )

    # Creating the prediction layer for the logits
    # of the decoder
    self.predict = nn.Linear(self.hidden_dim, self.output_dim)

  def encode(self, x):
    # Embedding x
    x_emb = self.encoder_embedding(x)

    # Getting the cell state and hidden states from encoding x
    # These h and c are also the initializations for the decoder
    encoder_output, (h, c) = self.encoder(x_emb)

    return encoder_output, (h, c)

  def decode(self, y_current, current_states):
    # Embedding the input for the decoder
    decoder_input_emb = self.decoder_embedding(y_current)

    # Passing the h and c states throught the decoder as well as the teacher forcing
    # decoder input
    decoder_output, current_states = self.decoder(decoder_input_emb, current_states)

    # Predicting the logits
    logits = self.predict(decoder_output)

    return logits[:, -1, :], current_states

  def forward(self, x, y_expected):
    # Embedding x and y_expected
    x_emb = self.encoder_embedding(x)
    y_expected_emb = self.decoder_embedding(y_expected)

    # Getting the cell state and hidden states from encoding x
    # These h and c are also the initializations for the decoder
    _, (h, c) = self.encoder(x_emb)

    # Subtracting 1 from the output length of y_expected since we are only predicting
    # up to the EOS token
    decoder_input = y_expected[:, :-1]
    # Embedding the input for the decoder
    decoder_input_emb = self.decoder_embedding(decoder_input)
    # Passing the h and c states throught the decoder as well as the teacher forcing
    # decoder input
    decoder_output, _ = self.decoder(decoder_input_emb, (h, c))

    # Predicting the logits
    batch_output = self.predict(decoder_output)

    return batch_output

In [ ]:
# The training and validation loop
# Not calculating the validation accuracies and no autoregressive decoding
# to speed up training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_validate_LSTM(model, train_loader, validation_loader, epochs, lr):
  model.to(device)

  # Using the Adam Optimizer
  optimizer = optim.Adam(model.parameters(), lr=lr)

  # Using the Categorical Cross Entropy Loss
  # Ignoring the padding tokens, which are "1"
  CCELoss = nn.CrossEntropyLoss(ignore_index=1)

  training_losses = []
  validation_losses = []

  for epoch in range(epochs):
    # Training:
    model.train()
    total_training_loss = 0

    for buggy, fixed in train_loader:
      buggy = buggy.to(device)
      fixed = fixed.to(device)

      optimizer.zero_grad()

      # Forward pass
      predicted_fixed = model.forward(buggy, fixed)

      # Calculating the loss
      # LLM used here for help with the reshaping (.view calls)
      loss = CCELoss(predicted_fixed.view(-1, model.output_dim), fixed[:, 1:].contiguous().view(-1))
      total_training_loss += loss.item()

      # Backpropagation
      loss.backward()

      # Optimizer step
      optimizer.step()

    # Printing the average training loss for that epoch:
    average_training_loss = total_training_loss/len(train_loader)
    training_losses.append(average_training_loss)
    print(f"Epoch {epoch + 1}:")
    print(f"Average Training Loss = {average_training_loss:.4f}")

    # Validation:
    model.eval()
    total_validation_loss = 0

    with torch.no_grad():
      for buggy, fixed in validation_loader:
        buggy = buggy.to(device)
        fixed = fixed.to(device)

        # Forward pass
        predicted_fixed = model.forward(buggy, fixed)

        # Calculating the loss
        loss = CCELoss(predicted_fixed.view(-1, model.output_dim), fixed[:, 1:].contiguous().view(-1))
        total_validation_loss += loss.item()

    # Printing the average validation loss for that epoch
    average_validation_loss = total_validation_loss/len(validation_loader)
    validation_losses.append(average_validation_loss)
    print(f"Average Validation Loss = {average_validation_loss:.4f}")
    print()

  # Plotting the training and validation loss plots
  plt.figure(figsize=(6, 8))

  plt.subplot(2, 1, 1)
  plt.plot([(i+1) for i in range(epochs)], training_losses, color='red')
  plt.ylabel("Loss")
  plt.ylim(bottom=0)
  plt.title("Average Training Loss Plot")
  plt.grid(True, alpha=0.3)

  plt.subplot(2, 1, 2)
  plt.plot([(i+1) for i in range(epochs)], validation_losses, color='blue')
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.ylim(bottom=0)
  plt.title("Average Validation Loss Plot")
  plt.grid(True, alpha=0.3)

  plt.suptitle(f"Training and Validation Loss Plots\nEpochs = {epochs}, Learning Rate = {lr}")
  plt.show()

In [ ]:
# Defining the greedy decoding function

def greedy_decode_no_attention(model, sequence, current_states):
  # Getting the logits and recurrent states for the next token
  logits, next_states = model.decode(sequence, current_states)
  # Predicting the next token in a greedy fashion
  token = torch.argmax(logits, dim=-1, keepdim=True)
  return token, next_states

In [ ]:
# The testing loop
# Using greedy decoding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def test_LSTM(model, test_loader):
  model.to(device)

  # Testing:
  model.eval()
  total_non_padding_tokens = 0
  correct_predictions_token = 0
  correct_predictions_sequence = 0

  with torch.no_grad():
    for buggy, fixed in test_loader:
      buggy = buggy.to(device)
      fixed = fixed.to(device)

      batch_size = buggy.size(0)
      target_seq_len = fixed.size(1)

      # Encoding
      encoder_output, current_states = model.encode(buggy)
      # Initializing the predicted sequence with the BOS token, which is "2" here
      current_token = torch.full((batch_size, 1), 2, dtype=torch.long, device=device)
      sequence = [current_token]

      # Autoregressively generating the predicted sequence after the BOS token
      for i in range(target_seq_len-1):
        predicted_token, current_states = greedy_decode_no_attention(model, current_token, current_states)
        sequence.append(predicted_token)
        current_token = predicted_token

      sequence = torch.cat(sequence, dim=1)

      # Calculating the accuracies based off of matches only with the
      # non-padding tokens

      # LLM used here for help with creating the non-padding-tokens mask
      # and calculating the accuracies
      # Creating the mask for finding the padding and non-padding tokens
      non_pad_mask = (fixed != 1)
      # Updating the total non-padding tokens
      total_non_padding_tokens += non_pad_mask.sum().item()

      # Getting the predictions, comparing them to the fixed code encodings
      # and updating the correct token-level predictions
      correct_token_matches = (sequence == fixed) & non_pad_mask
      correct_predictions_token += correct_token_matches.sum().item()

      # Comparing the entire sequences of predicted and fixed code encodings
      # and updating the correct sequence level predictions
      sequence_matches = torch.all(correct_token_matches | ~non_pad_mask, dim=1)
      correct_predictions_sequence += sequence_matches.sum().item()

  test_accuracy_token = (correct_predictions_token/total_non_padding_tokens)*100
  test_accuracy_sequence = (correct_predictions_sequence/len(test_loader.dataset))*100

  # Printing the token-level and exact-match accuracies
  print(f"Test Accuracy (Token-level) = {test_accuracy_token:.4f}%")
  print(f"Test Accuracy (Exact-Match) = {test_accuracy_sequence:.4f}%")

In [ ]:
# Initializing the LSTM Model
EPOCHS = 100
LEARNING_RATE = 0.002
VOCAB_SIZE = vocab_size
EMBEDDING_DIM = 256
HIDDEN_DIM = 128
model_lstm = LSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, VOCAB_SIZE)

# Training it
# train_and_validate_LSTM(model_lstm, train_loader, val_loader, EPOCHS, LEARNING_RATE)

In [ ]:
# Loading the trained checkpoints of the model
model_lstm.load_state_dict(torch.load('/content/Learning_Phase/Task-1/Subtask-2/Trained_Checkpoints/learningphase_task1_subtask2_lstm.pth'))

In [ ]:
# Testing the LSTM
test_LSTM(model_lstm, test_loader)

In [ ]:
# Saving the trained checkpoints
# torch.save(model_lstm.state_dict(), 'learningphase_task1_subtask2_lstm.pth')

In [ ]:
# Loading the transformer

tokenizer = PreTrainedTokenizerFast.from_pretrained("/content/Learning_Phase/Task-1/Subtask-2/./code_refinement_tokenizer")

In [ ]:
# Qualitative Output Demo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Check the expected vs generated fixed code
# for a given index in the test dataset
check_index = 0

check_sequence = X_test[check_index]
check_sequence_text = tokenizer.decode(check_sequence)
print("Test buggy code:")
print(check_sequence_text)

expected_fixed_code = y_test[check_index]
expected_fixed_code_text = tokenizer.decode(expected_fixed_code)
print()
print("Expected fixed code:")
print(expected_fixed_code_text)

buggy = torch.tensor(check_sequence).long().unsqueeze(0)
buggy = buggy.to(device)

batch_size = buggy.size(0)
target_seq_len = buggy.size(1)

model_lstm = model_lstm.to(device)

model_lstm.eval()
with torch.no_grad():
  # Encoding
  encoder_output, current_states = model_lstm.encode(buggy)
  # Initializing the predicted sequence with the BOS token, which is "2" here
  current_token = torch.full((batch_size, 1), 2, dtype=torch.long, device=device)
  sequence = [current_token]

  # Autoregressively generating the predicted sequence after the BOS token
  for i in range(target_seq_len-1):
    predicted_token, current_states = greedy_decode_no_attention(model_lstm, current_token, current_states)
    sequence.append(predicted_token)
    current_token = predicted_token

  sequence = torch.cat(sequence, dim=1)

sequence = sequence.squeeze()
sequence_text = tokenizer.decode(sequence)
print()
print("Generated fixed code:")
print(sequence_text)

##**Improving the LSTM Architecture:**
Adding an attention mechanism to the LSTM Encoder-Decoder architecture

In [ ]:
# Attention LSTM Architecture

class LSTM_Attention(nn.Module):
  def __init__(self, vocab_size, attention_heads, embedding_dim, hidden_dim, output_dim):
    super().__init__()

    # Initializing the LSTM parameters
    self.vocab_size = vocab_size
    self.attention_heads = attention_heads
    self.embedding_dim = embedding_dim
    self.hidden_dim = hidden_dim
    self.output_dim = output_dim

    # Embedding layer for the encoder to project the input to the embedding space
    self.encoder_embedding = nn.Embedding(self.vocab_size, self.embedding_dim)

    # Using a bidirectional, 2-layer encoder LSTM
    self.encoder = nn.LSTM(
        input_size=self.embedding_dim,
        hidden_size=self.hidden_dim,
        num_layers=2,
        # Adding a dropout of 0.3
        dropout=0.3,
        batch_first=True,
        bidirectional=True
    )

    # Reverse projecting the h and c states from the 2*hidden_dim dimesnions
    # to the hidden_dim dimensions
    self.h_projection = nn.Linear(2*self.hidden_dim, self.hidden_dim)
    self.c_projection = nn.Linear(2*self.hidden_dim, self.hidden_dim)

    # Embedding layer for the decoder to project the expected output to the embedding space
    self.decoder_embedding = nn.Embedding(self.vocab_size, self.embedding_dim)

    # Creating the 2-layer decoder LSTM
    self.decoder = nn.LSTM(
        input_size=self.embedding_dim,
        hidden_size=self.hidden_dim,
        num_layers=2,
        # Adding a dropout of 0.3
        dropout=0.3,
        batch_first=True,
        bidirectional=False
    )

    # Creating the Multi-Head Global Attention Block
    self.dk = (self.hidden_dim // self.attention_heads)

    self.q_projection = nn.Linear(self.hidden_dim, self.hidden_dim)
    self.k_projection = nn.Linear(2*self.hidden_dim, self.hidden_dim)
    self.v_projection = nn.Linear(2*self.hidden_dim, self.hidden_dim)

    self.multihead_rev_projection = nn.Linear(self.hidden_dim, self.hidden_dim)
    self.multihead_attention_norm = nn.LayerNorm(self.hidden_dim)

    # Creating the prediction layer for the logits
    # of the decoder
    self.predict = nn.Linear(self.hidden_dim, self.output_dim)

  # Creating a positional encoding fuction
  # LLM used here for help with the tensor-shape syntax
  def pos_encoding(self, seq_len, device):
    positional_encodings = torch.zeros(seq_len, self.hidden_dim, device=device)
    pos = torch.arange(0, seq_len, device=device).float().unsqueeze(1)
    inv_denominator = torch.exp(torch.arange(0, self.hidden_dim, 2, device=device).float() * (-math.log(10000.0) / self.hidden_dim))
    positional_encodings[:, 0::2] = torch.sin(pos * inv_denominator)
    positional_encodings[:, 1::2] = torch.cos(pos * inv_denominator)
    return positional_encodings.unsqueeze(0)

  def encode(self, x):
    # Calculating the batch size to use for reshaping later
    batch_size = x.size(0)
    # Creating the padding mask for the input
    mask = (x != 1).unsqueeze(1).unsqueeze(2)

    # Embedding x
    x_emb = self.encoder_embedding(x)

    # Gettig the encoder output, hidden and cell states from the encoder
    # These h and c are also the initializations for the decoder
    encoder_output, (h, c) = self.encoder(x_emb)

    # Projecting h and c to hidden_dim dimensions
    h = h.view(2, 2, batch_size, self.hidden_dim).transpose(1, 2).contiguous().view(2, batch_size, 2 * self.hidden_dim)
    c = c.view(2, 2, batch_size, self.hidden_dim).transpose(1, 2).contiguous().view(2, batch_size, 2 * self.hidden_dim)
    h = self.h_projection(h)
    c = self.c_projection(c)

    return encoder_output, mask, (h, c)

  def decode(self, y_current, sequence_history, encoder_output, mask, current_states):
    # Calculating the batch size and sequence legth to use for reshaping later
    batch_size = y_current.size(0)
    seq_len = sequence_history.size(1)

    # Embedding the input for the decoder
    decoder_input_emb = self.decoder_embedding(y_current)
    # Passing the h and c states throught the decoder as well as the teacher forcing
    # decoder input
    decoder_output, current_states = self.decoder(decoder_input_emb, current_states)

    # Getting the positional encoding for the current_token
    pos_enc = self.pos_encoding(seq_len, y_current.device)
    current_pos_enc = pos_enc[:, -1:, :]
    # Adding the positional encodings
    decoder_output_pos_enc = decoder_output + current_pos_enc
    # Keeping the residual for adding later
    attention_residual = decoder_output_pos_enc

    # Projecting the input to the queries, keys and values
    q = self.q_projection(decoder_output_pos_enc)
    k = self.k_projection(encoder_output)
    v = self.v_projection(encoder_output)
    # Calculating the encoder output sequence length
    encoder_output_seq_len = encoder_output.size(1)
    # Reshaping the projections to parallelly pass them to the different heads
    q = q.view(batch_size, 1, self.attention_heads, self.dk).transpose(1, 2)
    k = k.view(batch_size, encoder_output_seq_len, self.attention_heads, self.dk).transpose(1, 2)
    v = v.view(batch_size, encoder_output_seq_len, self.attention_heads, self.dk).transpose(1, 2)

    # Calculating the Attention Scores
    attention_scores = (torch.matmul(q, k.transpose(2, 3))/math.sqrt(self.dk))
    # Applying the paddig mask
    attention_scores = attention_scores.masked_fill(mask == False, -1e9)
    # Calculating the Attentio Weights
    attention_weights = torch.softmax(attention_scores, dim=-1)
    attention_output = torch.matmul(attention_weights, v)

    # Projecting the attention outputs back to the hidden dimensions
    attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, 1, self.hidden_dim)
    attention_output = self.multihead_rev_projection(attention_output)

    # Adding and normalizing with the residual
    multihead_output = self.multihead_attention_norm(attention_output + attention_residual)

    # Predicting the logits
    logits = self.predict(multihead_output)

    return logits.squeeze(1), current_states

  def forward(self, x, y_expected):
    # Calculating the batch size to use for reshaping later
    batch_size = x.size(0)
    # Creatig the paddig mask for the input
    mask = (x != 1).unsqueeze(1).unsqueeze(2)

    # Embedding x
    x_emb = self.encoder_embedding(x)

    # Getting the packed output, cell state, and hidden states from encoding x
    # These h and c are also the initializations for the decoder
    encoder_output, (h, c) = self.encoder(x_emb)

    # Projecting h and c to hidden_dim dimensions
    h = h.view(2, 2, batch_size, self.hidden_dim).transpose(1, 2).contiguous().view(2, batch_size, 2 * self.hidden_dim)
    c = c.view(2, 2, batch_size, self.hidden_dim).transpose(1, 2).contiguous().view(2, batch_size, 2 * self.hidden_dim)
    h = self.h_projection(h)
    c = self.c_projection(c)

    # Subtracting 1 from the output length of y_expected since we are only predicting
    # up to the <END> token
    decoder_input = y_expected[:, :-1]
    seq_len = decoder_input.size(1)
    # Embedding the input for the decoder
    decoder_input_emb = self.decoder_embedding(decoder_input)
    # Passing the h and c states throught the decoder as well as the teacher forcing
    # decoder input
    decoder_output, _ = self.decoder(decoder_input_emb, (h, c))

    # Getting the positional encodings
    pos_enc = self.pos_encoding(seq_len, x.device)
    # Adding the positional encodings
    decoder_output_pos_enc = decoder_output + pos_enc
    # Keeping the residual for adding later
    attention_residual = decoder_output_pos_enc

    # Projecting the input to the queries, keys and values
    q = self.q_projection(decoder_output_pos_enc)
    k = self.k_projection(encoder_output)
    v = self.v_projection(encoder_output)
    # Calculating the encoder output sequence length
    encoder_output_seq_len = encoder_output.size(1)
    # Reshaping the projections to parallelly pass them to the different heads
    q = q.view(batch_size, seq_len, self.attention_heads, self.dk).transpose(1, 2)
    k = k.view(batch_size, encoder_output_seq_len, self.attention_heads, self.dk).transpose(1, 2)
    v = v.view(batch_size, encoder_output_seq_len, self.attention_heads, self.dk).transpose(1, 2)

    # Calculating the Attention Scores
    attention_scores = (torch.matmul(q, k.transpose(2, 3))/math.sqrt(self.dk))
    # Applying the paddig mask
    attention_scores = attention_scores.masked_fill(mask == False, -1e9)
    # Calculating the Attentio Weights
    attention_weights = torch.softmax(attention_scores, dim=-1)
    attention_output = torch.matmul(attention_weights, v)

    # Projecting the attention outputs back to the hidden dimensions
    attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
    attention_output = self.multihead_rev_projection(attention_output)

    # Adding and normalizing with the residual
    multihead_output = self.multihead_attention_norm(attention_output + attention_residual)

    # Predicting the logits
    batch_output = self.predict(multihead_output)

    return batch_output

In [ ]:
# Defining the greedy decoding function

def greedy_decode_attention(model, current_token, sequence, encoder_output, mask, current_states):
  # Getting the logits and recurret states for the next token
  logits, next_states = model.decode(current_token, sequence, encoder_output, mask, current_states)
  # Predicting the next token in a greedy fashion
  token = torch.argmax(logits, dim=-1, keepdim=True)
  return token, next_states

In [ ]:
# The testing loop
# Using greedy decoding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def test_LSTM_Attention(model, test_loader):
  model.to(device)

  # Testing:
  model.eval()
  total_non_padding_tokens = 0
  correct_predictions_token = 0
  correct_predictions_sequence = 0

  with torch.no_grad():
    for buggy, fixed in test_loader:
      buggy = buggy.to(device)
      fixed = fixed.to(device)

      batch_size = buggy.size(0)
      target_seq_len = fixed.size(1)

      # Encoding
      encoder_output, mask, current_states = model.encode(buggy)
      # Initializing the predicted sequence with the BOS token, which is "2" here
      current_token = torch.full((batch_size, 1), 2, dtype=torch.long, device=device)
      sequence = current_token

      # Autoregressively generating the predicted sequence after the BOS token
      for i in range(target_seq_len-1):
        predicted_token, current_states = greedy_decode_attention(model, current_token, sequence, encoder_output, mask, current_states)
        sequence = torch.cat((sequence, predicted_token), dim=1)
        current_token = predicted_token

      # Calculating the accuracies based off of matches only with the
      # non-padding tokens

      # LLM used here for help with creating the non-padding-tokens mask
      # and calculating the accuracies
      # Creating the mask for finding the padding and non-padding tokens
      non_pad_mask = (fixed != 1)
      # Updating the total non-padding tokens
      total_non_padding_tokens += non_pad_mask.sum().item()

      # Getting the predictions, comparing them to the fixed code encodings
      # and updating the correct token-level predictions
      correct_token_matches = (sequence == fixed) & non_pad_mask
      correct_predictions_token += correct_token_matches.sum().item()

      # Comparing the entire sequences of predicted and fixed code encodings
      # and updating the correct sequence level predictions
      sequence_matches = torch.all(correct_token_matches | ~non_pad_mask, dim=1)
      correct_predictions_sequence += sequence_matches.sum().item()

  test_accuracy_token = (correct_predictions_token/total_non_padding_tokens)*100
  test_accuracy_sequence = (correct_predictions_sequence/len(test_loader.dataset))*100

  # Printing the token-level and exact-match accuracies
  print(f"Test Accuracy (Token-level) = {test_accuracy_token:.4f}%")
  print(f"Test Accuracy (Sequence-level) = {test_accuracy_sequence:.4f}%")

In [ ]:
# Initializing the Attention LSTM Model
EPOCHS = 25
LEARNING_RATE = 0.002
VOCAB_SIZE = vocab_size
HEADS = 8
EMBEDDING_DIM = 256
HIDDEN_DIM = 128
model_lstm_attention = LSTM_Attention(VOCAB_SIZE, HEADS, EMBEDDING_DIM, HIDDEN_DIM, VOCAB_SIZE)

# Training it
# train_and_validate_LSTM(model_lstm_attention, train_loader, val_loader, EPOCHS, LEARNING_RATE)

In [ ]:
# Loading the trained checkpoints of the model
model_lstm_attention.load_state_dict(torch.load('/content/Learning_Phase/Task-1/Subtask-2/Trained_Checkpoints/learningphase_task1_subtask2_lstm_attention.pth'))

In [ ]:
# Testing the Attention LSTM
test_LSTM_Attention(model_lstm_attention, test_loader)

In [ ]:
# Saving the trained checkpoints
# torch.save(model_lstm_attention.state_dict(), 'learningphase_task1_subtask2_lstm_attention.pth')

In [ ]:
# Qualitative Output Demo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Check the expected vs generated fixed code
# for a given index in the test dataset
check_index = 0

check_sequence = X_test[check_index]
check_sequence_text = tokenizer.decode(check_sequence)
print("Test buggy code:")
print(check_sequence_text)

expected_fixed_code = y_test[check_index]
expected_fixed_code_text = tokenizer.decode(expected_fixed_code)
print()
print("Expected fixed code:")
print(expected_fixed_code_text)

buggy = torch.tensor(check_sequence).long().unsqueeze(0)
buggy = buggy.to(device)

batch_size = buggy.size(0)
target_seq_len = buggy.size(1)

model_lstm_attention = model_lstm_attention.to(device)

model_lstm_attention.eval()
with torch.no_grad():
  # Encoding
  encoder_output, mask, current_states = model_lstm_attention.encode(buggy)
  # Initializing the predicted sequence with the BOS token, which is "2" here
  current_token = torch.full((batch_size, 1), 2, dtype=torch.long, device=device)
  sequence = current_token

  # Autoregressively generating the predicted sequence after the BOS token
  for i in range(target_seq_len-1):
    predicted_token, current_states = greedy_decode_attention(model_lstm_attention, current_token, sequence, encoder_output, mask, current_states)
    sequence = torch.cat((sequence, predicted_token), dim=1)
    current_token = predicted_token

sequence = sequence.squeeze()
sequence_text = tokenizer.decode(sequence)
print()
print("Generated fixed code:")
print(sequence_text)

##**Transformer Architecture:**

In [ ]:
# Loading all the encoded text sets

X_train_transformer = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/train_transformer_buggy_encodings.npz')
X_train_transformer = X_train_transformer['train_transformer_buggy_encodings']

X_val_transformer = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/val_transformer_buggy_encodings.npz')
X_val_transformer = X_val_transformer['val_transformer_buggy_encodings']

X_test_transformer = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/test_transformer_buggy_encodings.npz')
X_test_transformer = X_test_transformer['test_transformer_buggy_encodings']

y_train_transformer = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/train_transformer_fixed_encodings.npz')
y_train_transformer = y_train_transformer['train_transformer_fixed_encodings']

y_val_transformer = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/val_transformer_fixed_encodings.npz')
y_val_transformer = y_val_transformer['val_transformer_fixed_encodings']

y_test_transformer = np.load('/content/Learning_Phase/Task-1/Subtask-2/Encodings/test_transformer_fixed_encodings.npz')
y_test_transformer = y_test_transformer['test_transformer_fixed_encodings']

In [ ]:
# Finding the vocab size and maximum sequence length for the architecture

vocab_size_transformer = int(max(X_train_transformer.max(), y_train_transformer.max())) + 1

In [ ]:
# Wrapping the text data into a PyTorch Dataset

class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return (torch.from_numpy(self.X[index]).long(), torch.from_numpy(self.y[index]).long())

# Creating the DataLoader instances for the training, validation and
# testing sets
train_loader_transformer = DataLoader(TextDataset(X_train_transformer, y_train_transformer), batch_size=256, shuffle=True, num_workers=2, pin_memory=True, prefetch_factor=2)
val_loader_transformer = DataLoader(TextDataset(X_val_transformer, y_val_transformer), batch_size=256, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)
test_loader_transformer = DataLoader(TextDataset(X_test_transformer, y_test_transformer), batch_size=256, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)

In [ ]:
# Implementing the Transformer Architecture

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Transformer(nn.Module):
  def __init__(self, max_seq_len, vocab_size, hidden_dim, heads, layers):
    super().__init__()

    # Initializing the model parameters
    self.max_seq_len = max_seq_len
    self.vocab_size = vocab_size
    self.hidden_dim = hidden_dim
    self.heads = heads
    self.layers = layers

    # Creating the projection layer for the sequences
    self.project_input = nn.Embedding(self.vocab_size, self.hidden_dim)
    self.project_output = nn.Embedding(self.vocab_size, self.hidden_dim)

    # Causal mask for self attention in the decoder
    self.causal_mask_y = torch.tril(torch.ones((max_seq_len-1, max_seq_len-1), device=device)).bool().unsqueeze(0).unsqueeze(1)

    # Implementing the positional encodings matrix
    # LLM used here for help with the syntax to create the matrix
    # Creating the matrix
    positional_encodings = torch.zeros(self.max_seq_len, self.hidden_dim)
    # Calculating the positions
    pos = torch.arange(0, self.max_seq_len).float().unsqueeze(1)
    # Calculating the denominator term
    inv_denominator = torch.exp(torch.arange(0, self.hidden_dim, 2).float()*(-math.log(10000)/self.hidden_dim))
    # Calculating the positional encodings
    positional_encodings[:, 0::2] = torch.sin(pos*inv_denominator)
    positional_encodings[:, 1::2] = torch.cos(pos*inv_denominator)
    positional_encodings = positional_encodings.unsqueeze(0)
    self.register_buffer('positional_encodings', positional_encodings)

    # Creating the Encoder Block
    self.dk = (self.hidden_dim // self.heads)

    self.encoder_block = nn.ModuleList(
        [nn.ModuleDict({
            'qkv' : nn.Linear(self.hidden_dim, 3*self.hidden_dim),
            'rev_projection' : nn.Linear(self.hidden_dim, self.hidden_dim),
            'attention_norm' : nn.LayerNorm(self.hidden_dim),
            'fc' : nn.Sequential(
                nn.Linear(self.hidden_dim, 4*self.hidden_dim),
                nn.ReLU(),
                nn.Linear(4*self.hidden_dim, self.hidden_dim),
            ),
            'fc_norm' : nn.LayerNorm(self.hidden_dim)
        }) for i in range(self.layers)]
    )

    # Creating the Decoder Block

    self.decoder_block = nn.ModuleList(
        [nn.ModuleDict({
            'qkv_self' : nn.Linear(self.hidden_dim, 3*self.hidden_dim),
            'rev_projection_self' : nn.Linear(self.hidden_dim, self.hidden_dim),
            'attention_norm_self' : nn.LayerNorm(self.hidden_dim),
            'q_cross' : nn.Linear(self.hidden_dim, self.hidden_dim),
            'kv_cross' : nn.Linear(self.hidden_dim, 2*self.hidden_dim),
            'rev_projection_cross' : nn.Linear(self.hidden_dim, self.hidden_dim),
            'attention_norm_cross' : nn.LayerNorm(self.hidden_dim),
            'fc' : nn.Sequential(
                nn.Linear(self.hidden_dim, 4*self.hidden_dim),
                nn.ReLU(),
                nn.Linear(4*self.hidden_dim, self.hidden_dim),
            ),
            'fc_norm' : nn.LayerNorm(self.hidden_dim)
        }) for i in range(self.layers)]
    )

    # Creating the final prediction layer
    self.predict = nn.Linear(self.hidden_dim, self.vocab_size)

  def encode(self, x):
    # Storing the batch size for reshaping later
    batch_size = x.shape[0]
    seq_len = x.shape[1]

    # Creating the padding mask
    mask_pad_x = (x != 1).unsqueeze(1).unsqueeze(2)

    # Projecting the input to the hidden dimensional space
    projected_x = self.project_input(x)

    # Adding the positional encodings
    projected_x_with_pos_encodings = projected_x + self.positional_encodings[:, :seq_len, :]
    flowing_x = projected_x_with_pos_encodings

    # Performing self-attention on the encoder input and passing it through the feed forward layers
    for block in self.encoder_block:
      # Keeping the residual for adding later
      multihead_residual = flowing_x

      # Projecting the input to the queries, keys and values
      # LLM used here for the reshaping syntax
      qkv = block['qkv'](flowing_x)
      q, k, v = torch.chunk(qkv, 3, dim=-1)
      # Reshaping the projections to parallelly pass them to the different heads
      q = q.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      k = k.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      v = v.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)

      # LLM used here for the transpose syntax
      # Calculating the Attention Scores
      attention_scores = (torch.matmul(q, k.transpose(2, 3))/math.sqrt(self.dk))
      # Applying the padding mask for x
      attention_scores = attention_scores.masked_fill(~mask_pad_x, -1e9)
      # Calculating the Attention Weights
      attention_weights = torch.softmax(attention_scores, dim=-1)
      attention_output = torch.matmul(attention_weights, v)

      # LLM used here for the reshaping syntax
      # Projecting the attention outputs back to the hidden dimensions
      attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
      attention_output = block['rev_projection'](attention_output)

      # Adding and normalizing with the residual
      multihead_output = block['attention_norm'](attention_output + multihead_residual)
      # Storing the new residual for adding later
      fully_connected_residual = multihead_output

      # Feeding the Multi-head Attention output to the Fully-Connected Layer
      fully_connected_output = block['fc'](multihead_output)
      # Adding and normalizing with the residual
      fully_connected_output = block['fc_norm'](fully_connected_output + fully_connected_residual)

      flowing_x = fully_connected_output

    return flowing_x, batch_size, seq_len, mask_pad_x

  # Creating a fuction to calculate and cache the cross-attention keys and values
  # from the encoder output
  def cross_kv_cache(self, encoder_output, batch_size, seq_len, encoder_mask_pad):
    k_cross = []
    v_cross = []
    for block in self.decoder_block:
      kv = block['kv_cross'](encoder_output)
      k, v = torch.chunk(kv, 2, dim=-1)
      k = k.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      v = v.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      k_cross.append(k)
      v_cross.append(v)

    return k_cross, v_cross, batch_size, seq_len, encoder_mask_pad

  def decode(self, cross_kv_cache, self_kv_cache, y_current_token, pos):
    # Projecting the output to the hidden-dimensional space
    projected_y = self.project_output(y_current_token)

    # Adding the positional encodings
    projected_y_with_pos_encodings = projected_y + self.positional_encodings[:, pos, :]
    flowing_y = projected_y_with_pos_encodings

    # The KV cache and the padding mask for the input
    k_cross_cached, v_cross_cached, batch_size, seq_len, mask_pad_x = cross_kv_cache

    # Performing self-attention on the output, then cross-attention with the encoder output
    # and passing it through the feed forward layers
    for i in range(self.layers):
      block = self.decoder_block[i]

      # Keeping the residual for adding later
      multihead_residual_self = flowing_y

      # Projecting the output to the queries, keys and values
      # LLM used here for the reshaping syntax
      qkv = block['qkv_self'](flowing_y)
      q_self, k_self, v_self = torch.chunk(qkv, 3, dim=-1)
      # Reshaping the projections to parallelly pass them to the different heads
      q_self = q_self.view(batch_size, 1, self.heads, self.dk).transpose(1, 2)
      k_self = k_self.view(batch_size, 1, self.heads, self.dk).transpose(1, 2)
      v_self = v_self.view(batch_size, 1, self.heads, self.dk).transpose(1, 2)

      # Getting the accumulated keys and values from the self-attention cache:
      if pos > 0:
        k_self_cached, v_self_cached = self_kv_cache[i]
        k_self = torch.cat((k_self_cached, k_self), dim=2)
        v_self = torch.cat((v_self_cached, v_self), dim=2)
      self_kv_cache[i] = (k_self, v_self)

      # LLM used here for the transpose syntax
      # Calculating the Attention Scores
      attention_scores_self = (torch.matmul(q_self, k_self.transpose(2, 3))/math.sqrt(self.dk))
      # Clculating the Attention Weights
      attention_weights_self = torch.softmax(attention_scores_self, dim=-1)
      attention_output_self = torch.matmul(attention_weights_self, v_self)

      # LLM used here for the reshaping syntax
      # Projecting the attention outputs back to the hidden dimensions
      attention_output_self = attention_output_self.transpose(1, 2).contiguous().view(batch_size, 1, self.hidden_dim)
      attention_output_self = block['rev_projection_self'](attention_output_self)

      # Adding and normalizing with the residual
      multihead_output_self = block['attention_norm_self'](attention_output_self + multihead_residual_self)
      # Storing the new residual for adding later
      multihead_residual_cross = multihead_output_self

      # Projecting the self-attention output in the decoder to the queries
      # Getting the encoder output keys and values from the cache
      q_cross = block['q_cross'](multihead_output_self)
      # LLM used here for the reshaping syntax
      # Reshaping the projections to parallelly pass them to the different heads
      q_cross = q_cross.view(batch_size, 1, self.heads, self.dk).transpose(1, 2)
      k_cross = k_cross_cached[i]
      v_cross = v_cross_cached[i]

      # LLM used here for the transpose syntax
      # Calculating the Attention Scores
      attention_scores_cross = (torch.matmul(q_cross, k_cross.transpose(2, 3))/math.sqrt(self.dk))
      # Applying the padding mask for x
      attention_scores_cross = attention_scores_cross.masked_fill(~mask_pad_x, -1e9)
      # Calculating the Attention Weights
      attention_weights_cross = torch.softmax(attention_scores_cross, dim=-1)
      attention_output_cross = torch.matmul(attention_weights_cross, v_cross)

      # LLM used here for the reshaping syntax
      # Projecting the attention outputs back to the hidden dimensions
      attention_output_cross = attention_output_cross.transpose(1, 2).contiguous().view(batch_size, 1, self.hidden_dim)
      attention_output_cross = block['rev_projection_cross'](attention_output_cross)

      # Adding and normalizing with the residual
      multihead_output_cross = block['attention_norm_cross'](attention_output_cross + multihead_residual_cross)
      # Storing the new residual for adding later
      fully_connected_residual = multihead_output_cross

      # Feeding the Multi-head Attention output to the Fully-Connected Layer
      fully_connected_output = block['fc'](multihead_output_cross)
      # Adding and normalizing with the residual
      fully_connected_output = block['fc_norm'](fully_connected_output + fully_connected_residual)

      flowing_y = fully_connected_output

    # Predicting the final logits
    predicted_ranks = self.predict(flowing_y)

    return predicted_ranks, self_kv_cache

  def forward(self, x, y):
    # Storing the batch size for reshaping later
    batch_size = x.shape[0]
    seq_len = x.shape[1]

    # Creating the padding mask for the input
    mask_pad_x = (x != 1).unsqueeze(1).unsqueeze(2)

    # Projecting the input to the hidden dimensional space
    projected_x = self.project_input(x)

    # Adding the positional encodings
    projected_x_with_pos_encodings = projected_x + self.positional_encodings[:, :seq_len, :]
    flowing_x = projected_x_with_pos_encodings

    # Performing self-attention on the encoder input and passing it through the feed forward layers
    for block in self.encoder_block:
      # Keeping the residual for adding later
      multihead_residual = flowing_x

      # Projecting the input to the queries, keys and values
      # LLM used here for the reshaping syntax
      qkv = block['qkv'](flowing_x)
      q, k, v = torch.chunk(qkv, 3, dim=-1)
      # Reshaping the projections to parallelly pass them to the different heads
      q = q.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      k = k.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      v = v.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)

      # LLM used here for the transpose syntax
      # Calculating the Attention Scores
      attention_scores = (torch.matmul(q, k.transpose(2, 3))/math.sqrt(self.dk))
      # Applying the padding mask for x
      attention_scores = attention_scores.masked_fill(~mask_pad_x, -1e9)
      # Calculating the Attention Weights
      attention_weights = torch.softmax(attention_scores, dim=-1)
      attention_output = torch.matmul(attention_weights, v)

      # LLM used here for the reshaping syntax
      # Projecting the attention outputs back to the hidden dimensions
      attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
      attention_output = block['rev_projection'](attention_output)

      # Adding and normalizing with the residual
      multihead_output = block['attention_norm'](attention_output + multihead_residual)
      # Storing the new residual for adding later
      fully_connected_residual = multihead_output

      # Feeding the Multi-head Attention output to the Fully-Connected Layer
      fully_connected_output = block['fc'](multihead_output)
      # Adding and normalizing with the residual
      fully_connected_output = block['fc_norm'](fully_connected_output + fully_connected_residual)

      flowing_x = fully_connected_output

    # Slicing y to remove the last element
    y = y[:, :-1]

    # Creating the padding mask for the expected output
    mask_pad_y = (y != 1).unsqueeze(1).unsqueeze(2)

    # Padding and causal mask together
    mask_pad_and_causal_y = mask_pad_y & self.causal_mask_y

    # Projecting the output to the hidden-dimensional space
    projected_y = self.project_output(y)

    # Adding the positional encodings
    projected_y_with_pos_encodings = projected_y + self.positional_encodings[:, :seq_len-1, :]
    flowing_y = projected_y_with_pos_encodings

    # Performing self-attention on the output, then cross-attention with the encoder output
    # and passing it through the feed forward layers
    for block in self.decoder_block:
      # Keeping the residual for adding later
      multihead_residual_self = flowing_y

      # Projecting the output to the queries, keys and values
      # LLM used here for the reshaping syntax
      qkv = block['qkv_self'](flowing_y)
      q_self, k_self, v_self = torch.chunk(qkv, 3, dim=-1)
      # Reshaping the projections to parallelly pass them to the different heads
      q_self = q_self.view(batch_size, seq_len-1, self.heads, self.dk).transpose(1, 2)
      k_self = k_self.view(batch_size, seq_len-1, self.heads, self.dk).transpose(1, 2)
      v_self = v_self.view(batch_size, seq_len-1, self.heads, self.dk).transpose(1, 2)

      # LLM used here for the transpose syntax
      # Calculating the Attention Scores
      attention_scores_self = (torch.matmul(q_self, k_self.transpose(2, 3))/math.sqrt(self.dk))
      # Applying the padding and causal mask for y
      attention_scores_self = attention_scores_self.masked_fill(~mask_pad_and_causal_y, -1e9)
      # Calculating the Attetion Weights
      attention_weights_self = torch.softmax(attention_scores_self, dim=-1)
      attention_output_self = torch.matmul(attention_weights_self, v_self)

      # LLM used here for the reshaping syntax
      # Projecting the attention outputs back to the hidden dimensions
      attention_output_self = attention_output_self.transpose(1, 2).contiguous().view(batch_size, seq_len-1, self.hidden_dim)
      attention_output_self = block['rev_projection_self'](attention_output_self)

      # Adding and normalizing with the residual
      multihead_output_self = block['attention_norm_self'](attention_output_self + multihead_residual_self)
      # Storing the new residual for adding later
      multihead_residual_cross = multihead_output_self

      # Projecting the self-attention output in the decoder to the queries
      # and the encoder output to the keys and values
      # LLM used here for the reshaping syntax
      q_cross = block['q_cross'](multihead_output_self)
      kv = block['kv_cross'](flowing_x)
      k_cross, v_cross = torch.chunk(kv, 2, dim=-1)
      # Reshaping the projections to parallelly pass them to the different heads
      q_cross = q_cross.view(batch_size, seq_len-1, self.heads, self.dk).transpose(1, 2)
      k_cross = k_cross.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)
      v_cross = v_cross.view(batch_size, seq_len, self.heads, self.dk).transpose(1, 2)

      # LLM used here for the transpose syntax
      # Calculating the Attention Scores
      attention_scores_cross = (torch.matmul(q_cross, k_cross.transpose(2, 3))/math.sqrt(self.dk))
      # Applying the padding mask for x
      attention_scores_cross = attention_scores_cross.masked_fill(~mask_pad_x, -1e9)
      # Calculating the Attentio Weights
      attention_weights_cross = torch.softmax(attention_scores_cross, dim=-1)
      attention_output_cross = torch.matmul(attention_weights_cross, v_cross)

      # LLM used here for the reshaping syntax
      # Projecting the attention outputs back to the hidden dimensions
      attention_output_cross = attention_output_cross.transpose(1, 2).contiguous().view(batch_size, seq_len-1, self.hidden_dim)
      attention_output_cross = block['rev_projection_cross'](attention_output_cross)

      # Adding and normalizing with the residual
      multihead_output_cross = block['attention_norm_cross'](attention_output_cross + multihead_residual_cross)
      # Storing the new residual for adding later
      fully_connected_residual = multihead_output_cross

      # Feeding the Multi-head Attention output to the Fully-Connected Layer
      fully_connected_output = block['fc'](multihead_output_cross)
      # Adding and normalizing with the residual
      fully_connected_output = block['fc_norm'](fully_connected_output + fully_connected_residual)

      flowing_y = fully_connected_output

    # Predicting the final logits
    predicted_ranks = self.predict(flowing_y)

    return predicted_ranks

In [ ]:
# The training and validation loop
# Not calculating the validation accuracies and no autoregressive decoding
# to speed up training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_validate_Transformer(model, train_loader, validation_loader, epochs, lr):
  model.to(device)

  # Using the Adam Optimizer
  optimizer = optim.Adam(model.parameters(), lr=lr)

  # Using the Categorical Cross Entropy Loss
  # Ignoring the padding tokens, which are "1"
  CCELoss = nn.CrossEntropyLoss(ignore_index=1)

  training_losses = []
  validation_losses = []

  for epoch in range(epochs):
    # Training:
    model.train()
    total_training_loss = 0

    for buggy, fixed in train_loader:
      buggy = buggy.to(device)
      fixed = fixed.to(device)

      optimizer.zero_grad()

      # Forward pass
      predicted_fixed = model.forward(buggy, fixed)

      # Calculating the loss
      # LLM used here for help with the reshaping (.view calls)
      loss = CCELoss(predicted_fixed.view(-1, model.vocab_size), fixed[:, 1:].contiguous().view(-1))
      total_training_loss += loss.item()

      # Backpropagation
      loss.backward()

      # Optimizer step
      optimizer.step()

    # Printing the average training loss for that epoch:
    average_training_loss = total_training_loss/len(train_loader)
    training_losses.append(average_training_loss)
    print(f"Epoch {epoch + 1}:")
    print(f"Average Training Loss = {average_training_loss:.4f}")

    # Validation:
    model.eval()
    total_validation_loss = 0

    with torch.no_grad():
      for buggy, fixed in validation_loader:
        buggy = buggy.to(device)
        fixed = fixed.to(device)

        # Forward pass
        predicted_fixed = model.forward(buggy, fixed)

        # Calculating the loss
        loss = CCELoss(predicted_fixed.view(-1, model.vocab_size), fixed[:, 1:].contiguous().view(-1))
        total_validation_loss += loss.item()

    # Printing the average validation loss for that epoch
    average_validation_loss = total_validation_loss/len(validation_loader)
    validation_losses.append(average_validation_loss)
    print(f"Average Validation Loss = {average_validation_loss:.4f}")
    print()

  # Plotting the training and validation loss plots
  plt.figure(figsize=(6, 8))

  plt.subplot(2, 1, 1)
  plt.plot([(i+1) for i in range(epochs)], training_losses, color='red')
  plt.ylabel("Loss")
  plt.ylim(bottom=0)
  plt.title("Average Training Loss Plot")
  plt.grid(True, alpha=0.3)

  plt.subplot(2, 1, 2)
  plt.plot([(i+1) for i in range(epochs)], validation_losses, color='blue')
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.ylim(bottom=0)
  plt.title("Average Validation Loss Plot")
  plt.grid(True, alpha=0.3)

  plt.suptitle(f"Training and Validation Loss Plots\nEpochs = {epochs}, Learning Rate = {lr}")
  plt.show()

In [ ]:
# Defining the greedy decoding function

def greedy_decode_transformer(model, encoder_output, cross_kv_cache, current_token, self_kv_cache, pos):
  # Getting the logits and recurrent states for the next token
  logits, updated_self_kv_cache = model.decode(cross_kv_cache, self_kv_cache, current_token, pos)
  # Predicting the next token in a greedy fashion
  predicted_token = torch.argmax(logits, dim=-1)
  return predicted_token, updated_self_kv_cache

In [ ]:
# The testing loop
# Using greedy decoding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def test_Transformer(model, test_loader):
  model.to(device)

  # Testing:
  model.eval()
  total_non_padding_tokens = 0
  correct_predictions_token = 0
  correct_predictions_sequence = 0

  with torch.no_grad():
    for buggy, fixed in test_loader:
      buggy = buggy.to(device)
      fixed = fixed.to(device)

      batch_size = buggy.size(0)
      target_seq_len = fixed.size(1)

      # Encoding
      encoder_output, batch_size, seq_len, encoder_mask_pad = model.encode(buggy)
      # Caching the keys and values for the cross-attention in the decoder
      cross_kv_cache = model.cross_kv_cache(encoder_output, batch_size, seq_len, encoder_mask_pad)
      # Initializing the predicted sequence with the BOS token, which is "2" here
      current_token = torch.full((batch_size, 1), 2, dtype=torch.long, device=device)
      # Initializing the cache for the keys and values for the self-attention
      # in the decoder
      self_kv_cache = {i: (None, None) for i in range(model.layers)}
      sequence = current_token

      # Autoregressively generating the predicted sequence after the BOS token
      for i in range(target_seq_len-1):
        predicted_token, self_kv_cache = greedy_decode_transformer(model, encoder_output, cross_kv_cache, current_token, self_kv_cache, i)
        sequence = torch.cat((sequence, predicted_token), dim=1)
        current_token = predicted_token

      # Calculating the accuracies based off of matches only with the
      # non-padding tokens

      # LLM used here for help with creating the non-padding-tokens mask
      # and calculating the accuracies
      # Creating the mask for finding the padding and non-padding tokens
      non_pad_mask = (fixed != 1)
      # Updating the total non-padding tokens
      total_non_padding_tokens += non_pad_mask.sum().item()

      # Getting the predictions, comparing them to the fixed code encodings
      # and updating the correct token-level predictions
      correct_token_matches = (sequence == fixed) & non_pad_mask
      correct_predictions_token += correct_token_matches.sum().item()

      # Comparing the entire sequences of predicted and fixed code encodings
      # and updating the correct sequence level predictions
      sequence_matches = torch.all(correct_token_matches | ~non_pad_mask, dim=1)
      correct_predictions_sequence += sequence_matches.sum().item()

  test_accuracy_token = (correct_predictions_token/total_non_padding_tokens)*100
  test_accuracy_sequence = (correct_predictions_sequence/len(test_loader.dataset))*100

  # Printing the token-level and exact-match accuracies
  print(f"Test Accuracy (Token-level) = {test_accuracy_token:.4f}%")
  print(f"Test Accuracy (Sequence-level) = {test_accuracy_sequence:.4f}%")

In [ ]:
# Initializing the Transformer Model
EPOCHS = 30
LEARNING_RATE = 0.0005
MAX_SEQ_LEN = X_train_transformer.shape[1]
VOCAB_SIZE = vocab_size_transformer
HIDDEN_DIM = 128
HEADS = 8
LAYERS = 4
model_transformer = Transformer(MAX_SEQ_LEN, VOCAB_SIZE, HIDDEN_DIM, HEADS, LAYERS)

# Training it
# train_and_validate_Transformer(model_transformer, training_history_transformer, validation_history_transformer, train_loader_transformer, val_loader_transformer, EPOCHS, LEARNING_RATE)

In [ ]:
# Loading the trained checkpoints of the model
model_transformer.load_state_dict(torch.load('/content/Learning_Phase/Task-1/Subtask-2/Trained_Checkpoints/learningphase_task1_subtask2_transformer.pth'))

In [ ]:
# Testing the Transformer
test_Transformer(model_transformer, test_loader_transformer)

In [ ]:
# Saving the trained checkpoints
# torch.save(model_transformer.state_dict(), 'learningphase_task1_subtask2_transformer.pth')

In [ ]:
# Qualitative Output Demo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Check the expected vs generated fixed code
# for a given index in the test dataset
check_index = 0

check_sequence = X_test_transformer[check_index]
check_sequence_text = tokenizer.decode(check_sequence)
print("Test buggy code:")
print(check_sequence_text)

expected_fixed_code = y_test_transformer[check_index]
expected_fixed_code_text = tokenizer.decode(expected_fixed_code)
print()
print("Expected fixed code:")
print(expected_fixed_code_text)

buggy = torch.tensor(check_sequence).long().unsqueeze(0)
buggy = buggy.to(device)

batch_size = buggy.size(0)
target_seq_len = buggy.size(1)

model_transformer = model_transformer.to(device)

model_transformer.eval()
with torch.no_grad():
  # Encoding
  encoder_output, batch_size, seq_len, encoder_mask_pad = model_transformer.encode(buggy)
  # Caching the keys and values for the cross-attention in the decoder
  cross_kv_cache = model_transformer.cross_kv_cache(encoder_output, batch_size, seq_len, encoder_mask_pad)
  # Initializing the predicted sequence with the BOS token, which is "2" here
  current_token = torch.full((batch_size, 1), 2, dtype=torch.long, device=device)
  # Initializing the cache for the keys and values for the self-attention
  # in the decoder
  self_kv_cache = {i: (None, None) for i in range(model_transformer.layers)}
  sequence = current_token

  # Autoregressively generating the predicted sequence after the BOS token
  for i in range(target_seq_len-1):
    predicted_token, self_kv_cache = greedy_decode_transformer(model_transformer, encoder_output, cross_kv_cache, current_token, self_kv_cache, i)
    sequence = torch.cat((sequence, predicted_token), dim=1)
    current_token = predicted_token

sequence = sequence.squeeze()
sequence_text = tokenizer.decode(sequence)
print()
print("Generated fixed code:")
print(sequence_text)